In [1]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path

In [2]:
# import train data 
df = pd.read_csv("/home/sjoon/projects/brain_connectivity_classifier/data/raw/PIOP2_restingstate.csv")
df.shape

(224, 26797)

In [3]:
# Extract connection columns
connection_columns = [col for col in df.columns if '~' in str(col)]

# Extract actual regions from connection columns
def extract_regions(connection_columns):
    unique_regions = []
    seen = set()
    
    for col in connection_columns:
        region_a, region_b = col.split('~', 1)
        for region in [region_a, region_b]:
            if region not in seen:
                seen.add(region)
                unique_regions.append(region)
    
    region_to_idx = {region: idx for idx, region in enumerate(unique_regions)}
    n_regions = len(unique_regions)
    
    return unique_regions, region_to_idx, n_regions

# Extract from your actual data
region_list, region_to_idx, n_regions = extract_regions(connection_columns)

# Print results
print(f"Found {n_regions} regions")
print(f"Sample regions: {region_list[:3]}")
print(f"Connection columns: {len(connection_columns)}")

Found 232 regions
Sample regions: ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1']
Connection columns: 26796


In [4]:
def reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions):
    n_subjects = df.shape[0]
    matrices = np.zeros((n_subjects, n_regions, n_regions))
    
    values = df[connection_columns].values
    
    for subj_idx in range(n_subjects):
        matrix = matrices[subj_idx]
        for col_idx, col in enumerate(connection_columns):
            region_a, region_b = col.split('~', 1)
            idx_a = region_to_idx[region_a]
            idx_b = region_to_idx[region_b]
            
            # Place value symmetrically
            value = values[subj_idx, col_idx]
            matrix[idx_a, idx_b] = value
            matrix[idx_b, idx_a] = value
        
        # Self-correlations
        np.fill_diagonal(matrix, 1.0)
    
    return matrices 

df_mat = reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions)
df_mat.shape

(224, 232, 232)

In [5]:
df_mat[0].shape

(232, 232)

In [6]:
# Extract diagonal and flatten
diagonal_values = np.diagonal(df_mat[0])

# first 30 diagonal values of subject 1
diagonal_values[0:30]


array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [7]:
# first row of subject 1
df_mat[0][0].shape

(232,)

In [8]:
# Export df_mat
np.save("/home/sjoon/projects/brain_connectivity_classifier/data/processed/df_mat.npy", df_mat)

### Impute diagonal with row mean of off diagonal values 

In [9]:
def region_mean(matrices):
    """Impute diagonal with row-wise mean (excluding diagonal)."""
    result = matrices.copy()
    n_subjects, n_regions, _ = matrices.shape
    
    mask = ~np.eye(n_regions, dtype=bool) # Create a mask for off-diagonal elements
    
    for s in range(n_subjects):
        for i in range(n_regions):                  # Iterate over rows
            off_diagonal = result[s, i, mask[i]]    # Get off-diagonal elements
            result[s, i, i] = np.mean(off_diagonal) # Replace diagonal with mean
    
    return result

# Pass the ENTIRE 3D array at once
df_mat_region_mean = region_mean(df_mat)

# Extract diagonal from first subject
diagonal_values_1 = np.diagonal(df_mat_region_mean[0])
diagonal_values_2 = np.diagonal(df_mat_region_mean[1])


# First 30 diagonal values of subject 0
print(diagonal_values_1[0:30])
print('')
print(diagonal_values_2[0:30])

[ 0.01459015  0.03607282  0.00807049 -0.00606681  0.01733707 -0.00624904
  0.03133134  0.01832182  0.02596562 -0.00612919 -0.04136876  0.0023576
  0.06359088 -0.0055964   0.06406231  0.07223399  0.07067154  0.05916045
  0.03830454  0.05045568  0.03723962  0.05972994  0.01936207  0.02171968
  0.00558312 -0.01173927  0.02679792  0.07845583  0.05245688  0.03620511]

[-0.00958387 -0.00302883  0.00438229 -0.03332292  0.00129335 -0.00699611
  0.01969541  0.01780487  0.02098982  0.0235895   0.01781147 -0.00747013
  0.07013748  0.01773246 -0.00736572 -0.02642481  0.04913203  0.06956098
  0.03077331  0.03372396  0.06259389  0.02383804  0.05355721  0.08090534
  0.05090883  0.07452571  0.05829836  0.05175911 -0.04265438 -0.0063667 ]


In [10]:
df_mat[1][0][1:].mean()  

np.float64(-0.0095838689559775)

## Preprocessing

### Fisher-z transformation

In [11]:
def fisher_z_transform(matrix, clip_value=0.999999):
    # Clip values to avoid infinities at r = ±1
    clipped_matrix = np.clip(matrix, -clip_value, clip_value)
    
    # Apply transformation to the CLIPPED matrix
    return np.arctanh(clipped_matrix)

# Apply fisher z transformation
df_mat_fz = np.array([fisher_z_transform(df_mat_region_mean[i]) for i in range(df_mat_region_mean.shape[0])])

# Extract diagonal from first subject
diagonal_values_1 = np.diagonal(df_mat_fz[0])
diagonal_values_2 = np.diagonal(df_mat_fz[1])

# First 30 diagonal values of subject 0
print(f'First 30 diagonal values of subject 0 \n {diagonal_values_1[0:30]}')
print('')
print(f'First 30 diagonal values of subject 1 \n {diagonal_values_1[0:30]}')


First 30 diagonal values of subject 0 
 [ 0.01459119  0.03608848  0.00807066 -0.00606688  0.0173388  -0.00624913
  0.0313416   0.01832387  0.02597146 -0.00612926 -0.04139239  0.00235761
  0.0636768  -0.00559645  0.06415016  0.07236001  0.07078955  0.05922961
  0.03832329  0.05049857  0.03725685  0.05980112  0.01936449  0.0217231
  0.00558318 -0.01173981  0.02680433  0.07861741  0.05250508  0.03622094]

First 30 diagonal values of subject 1 
 [ 0.01459119  0.03608848  0.00807066 -0.00606688  0.0173388  -0.00624913
  0.0313416   0.01832387  0.02597146 -0.00612926 -0.04139239  0.00235761
  0.0636768  -0.00559645  0.06415016  0.07236001  0.07078955  0.05922961
  0.03832329  0.05049857  0.03725685  0.05980112  0.01936449  0.0217231
  0.00558318 -0.01173981  0.02680433  0.07861741  0.05250508  0.03622094]


In [12]:
print(f'5 rows x 5 col of subject 0 \n {df_mat_fz[0][0:5, 0:5]} ')
print(f' \n 5 rows x 5 col of subject 1 \n {df_mat_fz[1][0:5, 0:5]} ')

5 rows x 5 col of subject 0 
 [[ 0.01459119  0.48374485  0.58212765  0.28411527  0.33268149]
 [ 0.48374485  0.03608848  0.92887857  0.54633774  0.59758176]
 [ 0.58212765  0.92887857  0.00807066  0.63434606  0.33372215]
 [ 0.28411527  0.54633774  0.63434606 -0.00606688  0.69087258]
 [ 0.33268149  0.59758176  0.33372215  0.69087258  0.0173388 ]] 
 
 5 rows x 5 col of subject 1 
 [[-0.00958416  0.41619344  0.20542718  0.54997872  0.5313618 ]
 [ 0.41619344 -0.00302884  0.41847708  0.42191381  0.86791993]
 [ 0.20542718  0.41847708  0.00438232  0.58216001  0.30742853]
 [ 0.54997872  0.42191381  0.58216001 -0.03333526  0.74325942]
 [ 0.5313618   0.86791993  0.30742853  0.74325942  0.00129335]] 


### Modelling

In [14]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Prepare indices
np.random.seed(42)
X = df_mat_fz.reshape(-1, 232)
y = np.tile(np.arange(232), 224)
groups = np.repeat(np.arange(224), 232)

print(f"Shape of df_mat_fz: {df_mat_fz.shape}")
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"Shape of groups: {groups.shape}")

gkf = GroupKFold(n_splits=3)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    # Split data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Fit scaler on training data only, then transform both
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Train
    model = LogisticRegression(
        C=0.0343304473310619,
        max_iter=1000,
        solver='saga',
        multi_class='multinomial'
    )
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    score = accuracy_score(y_val, model.predict(X_val_scaled))

    # Store
    fold_scores.append(score)
    print(f"Fold {fold + 1} Accuracy: {score:.4f}")

print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")


Shape of df_mat_fz: (224, 232, 232)
Shape of X: (51968, 232)
Shape of y: (51968,)
Shape of groups: (51968,)


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 1 Accuracy: 0.9209


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 2 Accuracy: 0.9117


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 3 Accuracy: 0.9129

Mean CV Accuracy: 0.9152 ± 0.0041
